# Pull matching sequences from the petadex PostgreSQL `fastaa` table

**Workflow:** upload `petadex_intensity_2026-07-16.csv` -> connect to Postgres once and pull only the
rows matching your `gene_id`s (via `WHERE accession = ANY(...)`, not a full table dump) -> cache the
result to Drive -> on every future run, load from that Drive cache instead of reconnecting to the
database at all. A separate filtering cell lets you slice the cached data further with real SQL
syntax (via DuckDB, running locally against the cached data) without ever touching the DB connection
again.

**Security note:** the password field below uses `getpass` (not echoed to the screen, not saved in
the notebook file) rather than a hardcoded string in a cell -- worth keeping that way even for solo
use, since notebooks have a way of ending up shared/uploaded/committed without you fully meaning to.

## 1. Setup

In [ ]:
!pip install -q psycopg2-binary duckdb

import os
import pandas as pd
import duckdb
import psycopg2
from getpass import getpass

from google.colab import drive
from google.colab import files
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/petalytics/labels'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CACHE_PATH = os.path.join(OUTPUT_DIR, 'fastaa_matched.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Upload and load the intensity CSV
Upload `petadex_intensity_2026-07-16.csv` via the file icon in the left sidebar (or drag-and-drop),
then run this cell.

In [ ]:
intensity_df = pd.read_csv('petadex_intensity_2026-07-16.csv')
target_ids = sorted(intensity_df['accession'].dropna().unique().tolist())
print(f'{len(target_ids)} unique accessions to match against fastaa.')
print(target_ids)

254 unique accessions to match against fastaa.
['ADV92526.1', 'AR241010_100', 'AR241010_106', 'AR241010_11', 'AR241010_12', 'AR241010_13', 'AR241010_134', 'AR241010_14', 'AR241010_142', 'AR241010_152', 'AR241010_152_M1', 'AR241010_152_M2', 'AR241010_152_M3', 'AR241010_152_M4', 'AR241010_154', 'AR241010_169', 'AR241010_169_M1', 'AR241010_17', 'AR241010_173', 'AR241010_33', 'AR241010_43', 'AR241010_56', 'AR241010_73', 'AR241010_74', 'AR241010_77', 'AR241010_8', 'AR241010_86', 'ASA57064.1', 'BAB86909.1', 'BAC67195.1', 'BAF57210.1', 'BAN42606.1', 'DRR016455_58', 'DRR046399_6024', 'DRR056885_121', 'DRR322242_2006', 'DRR378781_602206', 'DRR509554_23646', 'ERR10087380_58920', 'ERR10123727_166025', 'ERR10482439_252531', 'ERR10482440_903875', 'ERR10640994_177560', 'ERR10640994_226214', 'ERR10640994_379232', 'ERR10641118_142160', 'ERR10641118_502282', 'ERR10789850_92482', 'ERR10822306_273555', 'ERR11028298_3008807', 'ERR11028298_3008807_M1', 'ERR11028393_217052', 'ERR11484209_22782', 'ERR1211564

## 3. Database connection settings
Fill in host/port/dbname/user below. Password is prompted securely when this cell runs -- it won't
appear in the notebook's saved output.

In [ ]:
DB_HOST = 'petadex.ccz9y6yshbls.us-east-1.rds.amazonaws.com'
DB_PORT = 5432
DB_NAME = 'petadex'
DB_USER = 'readonly_user'
DB_PASSWORD = getpass('Database password:')

FORCE_REFRESH = True  # set True to bypass the Drive cache and re-query the database

Database password:··········


## 4. Pull matching rows -- only if no cache exists yet (or FORCE_REFRESH is set)
This is the only cell in the whole notebook that touches the database. Everything after this reads
from the Drive-cached CSV, so reopening this notebook tomorrow doesn't require reconnecting at all
unless you deliberately set `FORCE_REFRESH = True` above.

In [ ]:
if os.path.exists(CACHE_PATH) and not FORCE_REFRESH:
    fastaa_df = pd.read_csv(CACHE_PATH)
    print(f'Loaded {len(fastaa_df)} rows from Drive cache -- no database connection made.')
    print(f'(Cache: {CACHE_PATH}. Set FORCE_REFRESH = True above to re-query the database instead.)')
else:
    print('Connecting to the database...')
    conn = psycopg2.connect(
        host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD,
    )
    query = '''
        SELECT accession, aa_sequence, source, synthetic
        FROM fastaa
        WHERE accession = ANY(%(ids)s)
    '''
    cur = conn.cursor()
    cur.execute(query, {'ids': target_ids})
    colnames = [desc[0] for desc in cur.description]
    rows = cur.fetchall()
    fastaa_df = pd.DataFrame(rows, columns=colnames)
    cur.close()
    conn.close()
    print('Connection closed.')

    fastaa_df.to_csv(CACHE_PATH, index=False)
    print(f'Saved {len(fastaa_df)} rows to Drive cache: {CACHE_PATH}')

Connecting to the database...
Connection closed.
Saved 254 rows to Drive cache: /content/drive/MyDrive/petalytics/labels/fastaa_matched.csv


## 5. Quick diagnostics before trusting this

In [ ]:
matched = set(fastaa_df['accession'])
missing = [i for i in target_ids if i not in matched]
print(f'{len(matched)}/{len(target_ids)} gene_ids matched in fastaa.')
if missing:
    print(f'{len(missing)} unmatched (first 10):', missing[:10])

print()
print('source value counts:')
print(fastaa_df['source'].value_counts(dropna=False))
print()
print('synthetic value counts:')
print(fastaa_df['synthetic'].value_counts(dropna=False))
print()
print('Sequence length stats:')
print(fastaa_df['aa_sequence'].str.len().describe())
fastaa_df.head()

254/254 gene_ids matched in fastaa.

source value counts:
source
Logan 1.0        180
manual design     37
GRASP_241010      21
GenBank           16
Name: count, dtype: int64

synthetic value counts:
synthetic
False    196
True      58
Name: count, dtype: int64

Sequence length stats:
count    254.000000
mean     336.996063
std       73.916441
min      134.000000
25%      290.000000
50%      308.000000
75%      392.250000
max      575.000000
Name: aa_sequence, dtype: float64


,accession,aa_sequence,source,synthetic
0,G9BY57.1,MDGVLWRVRTAALMAALLALAAWALVWASPSVEAQSNPYQRGPNPT...,GenBank,False
1,SRR26075152_134008,MNVNTLRRRVLRVATGVGLLALSFATRAQTVVFQEAFANGLGQFTS...,Logan 1.0,False
2,SRR26290402_3525830,MRTAKDFARGNPLARLCILLALAVFPALSHAQTVIFQEPFNVGIGS...,Logan 1.0,False
3,SRR26290450_3599995,MNFKTSIARTALLVASLVASSYVYSATGPSAPCTNCTRGPAPTAAA...,Logan 1.0,False
4,SRR26290483_118036,MRTRGTRPLRRLAAIFLLALVIAACDTKGPDPTEQILTASSGPFGV...,Logan 1.0,False


## 6. Further filtering with real SQL -- runs locally, no database connection
Uses DuckDB to run SQL directly against the `fastaa_df` DataFrame already in memory (loaded from the
Drive cache above). Edit `SQL_QUERY` however you like -- standard SQL syntax, `fastaa_df` is treated
as a table name.

In [ ]:
SQL_QUERY = '''
    SELECT *
    FROM fastaa_df
    WHERE aa_sequence IS NOT NULL
      AND LENGTH(aa_sequence) > 0
'''

filtered_df = duckdb.sql(SQL_QUERY).df()
print(f'{len(filtered_df)}/{len(fastaa_df)} rows after filtering.')
filtered_df.head()

254/254 rows after filtering.


,accession,aa_sequence,source,synthetic
0,G9BY57.1,MDGVLWRVRTAALMAALLALAAWALVWASPSVEAQSNPYQRGPNPT...,GenBank,False
1,SRR26075152_134008,MNVNTLRRRVLRVATGVGLLALSFATRAQTVVFQEAFANGLGQFTS...,Logan 1.0,False
2,SRR26290402_3525830,MRTAKDFARGNPLARLCILLALAVFPALSHAQTVIFQEPFNVGIGS...,Logan 1.0,False
3,SRR26290450_3599995,MNFKTSIARTALLVASLVASSYVYSATGPSAPCTNCTRGPAPTAAA...,Logan 1.0,False
4,SRR26290483_118036,MRTRGTRPLRRLAAIFLLALVIAACDTKGPDPTEQILTASSGPFGV...,Logan 1.0,False


## 7. Optional: merge back with the intensity data
Joins the filtered sequences back onto the original intensity rows, for a single combined view.
Skip this cell if you'd rather keep sequences and intensity data separate.

In [ ]:
combined_df = intensity_df.merge(
    filtered_df, left_on='accession', right_on='accession', how='inner',
)
print(f'{len(combined_df)} combined rows ({combined_df["gene_id"].nunique()} unique gene_ids).')
combined_df.head()

3084 combined rows (257 unique gene_ids).


,gene_id,nickname,accession,source_x,substrate,timepoint_hours,average_intensity,stddev_intensity,sample_count,aa_sequence,source_y,synthetic
0,AdCut,TragicAttitude,WP_058088494.1,GenBank,BHET12.5,8,-0.005689,0.012097,8,MKNAPIRSVLATCLVAAAASALAQTTAPTSASLNASAGPLSVSTAS...,GenBank,False
1,AdCut,TragicAttitude,WP_058088494.1,GenBank,BHET12.5,24,-0.003632,0.003080,8,MKNAPIRSVLATCLVAAAASALAQTTAPTSASLNASAGPLSVSTAS...,GenBank,False
2,AdCut,TragicAttitude,WP_058088494.1,GenBank,BHET12.5,48,0.004064,0.021105,8,MKNAPIRSVLATCLVAAAASALAQTTAPTSASLNASAGPLSVSTAS...,GenBank,False
3,AdCut,TragicAttitude,WP_058088494.1,GenBank,BHET12.5,72,0.008745,0.009014,8,MKNAPIRSVLATCLVAAAASALAQTTAPTSASLNASAGPLSVSTAS...,GenBank,False
4,AdCut,TragicAttitude,WP_058088494.1,GenBank,BHET12.5,96,0.004397,0.019748,8,MKNAPIRSVLATCLVAAAASALAQTTAPTSASLNASAGPLSVSTAS...,GenBank,False


##8. More filtering

In [ ]:
if 'combined_df' not in dir():
    raise NameError('combined_df not found -- run section 7 (merge) first.')

SQL_QUERY_COMBINED = '''
    SELECT *
    FROM combined_df
    WHERE 1=1
    AND substrate != 'BHET12.5'
    AND timepoint_hours = '120'
'''

query_result_df = duckdb.sql(SQL_QUERY_COMBINED).df()
print(f'{len(query_result_df)}/{len(combined_df)} rows returned.')
display(query_result_df.head(20))

EXPORT_RESULT = True  # flip to True and rerun this cell to actually save/download
EXPORT_FILENAME = 'combined_query_result.csv'  # change per-query if you want to keep multiple exports

if EXPORT_RESULT:
    export_path = os.path.join(OUTPUT_DIR, EXPORT_FILENAME)
    query_result_df.to_csv(export_path, index=False)
    print('Saved:', export_path)
    files.download(export_path)
else:
    print('EXPORT_RESULT is False -- not saving. Set to True above and rerun this cell to export.')

257/3084 rows returned.


,gene_id,nickname,accession,source_x,substrate,timepoint_hours,average_intensity,stddev_intensity,sample_count,aa_sequence,source_y,synthetic
0,AdCut,TragicAttitude,WP_058088494.1,GenBank,BHET25,120,-0.003304,0.027719,8,MKNAPIRSVLATCLVAAAASALAQTTAPTSASLNASAGPLSVSTAS...,GenBank,False
1,AR00001,UnimportantConsist,AR241010_8,GRASP_241010,BHET25,120,0.120480,0.072383,8,MKTSRHSTATTTLTRLARLALAAAAVLLSAAAQAQVVIFQENFNSG...,GRASP_241010,True
2,AR00002,UntimelyInevitable,AR241010_11,GRASP_241010,BHET25,120,0.085387,0.017821,8,MKTSRTRTATTTLTRLARLALAAAALLLSAAAQAQSSNPYQRGPDP...,GRASP_241010,True
3,AR00003,PleasingReference,AR241010_12,GRASP_241010,BHET25,120,0.153507,0.042544,8,MKTSRTRRLALAAAALLLSVAAQAQSSNPYQRGPDPTAASLEASSG...,GRASP_241010,True
4,AR00004,IllSale,AR241010_13,GRASP_241010,BHET25,120,0.000574,0.033122,8,MKTNRRRRLALAAAALLLSVAAQAQNGSAPPGPDPDPDPNPPESDN...,GRASP_241010,True
5,AR00005,CompleteSurprise,AR241010_14,GRASP_241010,BHET25,120,-0.023333,0.004384,8,MKTNRRKKLALAASALLLSMSAFATNPSPDPDPDPDPNPCESDCGY...,GRASP_241010,True
6,AR00006,GreatOpening,AR241010_17,GRASP_241010,BHET25,120,0.021365,0.034074,8,MKTTRLKKLALAASALLFSMSAFSFNPSPDPDPDPDPNPCQSDCGF...,GRASP_241010,True
7,AR00007,UnkemptConfusion,AR241010_33,GRASP_241010,BHET25,120,-0.030022,0.004522,8,MKTTTFKKSALSALTLLFSLSAFSGGGGGDDGNNDQLPAPDTCNSN...,GRASP_241010,True
8,AR00008,LightTeaching,AR241010_43,GRASP_241010,BHET25,120,-0.034515,0.003104,8,MNTTRLKKSALSLLAAGALLFSASAFATNPPPDPDPGPNPCESDCG...,GRASP_241010,True
9,AR00009,KookyMirror,AR241010_56,GRASP_241010,BHET25,120,0.055951,0.038832,8,MKTNMLTKLALAASALLLSMSVFATTPSPGPEPDPDPSPCSSSNGY...,GRASP_241010,True


Saved: /content/drive/MyDrive/petalytics/labels/combined_query_result.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Export

In [ ]:
from google.colab import files

filtered_df.to_csv(os.path.join(OUTPUT_DIR, 'fastaa_filtered.csv'), index=False)
combined_df.to_csv(os.path.join(OUTPUT_DIR, 'fastaa_intensity_combined.csv'), index=False)
print('Saved to Drive:')
print(' ', os.path.join(OUTPUT_DIR, 'fastaa_filtered.csv'))
print(' ', os.path.join(OUTPUT_DIR, 'fastaa_intensity_combined.csv'))

files.download(os.path.join(OUTPUT_DIR, 'fastaa_intensity_combined.csv'))

Saved to Drive:
  /content/drive/MyDrive/petalytics/labels/fastaa_filtered.csv
  /content/drive/MyDrive/petalytics/labels/fastaa_intensity_combined.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>